# Train/Test 문서 이미지 EDA 파이프라인

GPU 서버 장애로 산출물이 사라졌을 때도 동일한 지표로 빠르게 분포를 재구성하고, 학습/추론 전략을 즉시 조정할 수 있도록 설계한 노트북입니다. Train과 Test의 공통 지표를 같은 bin과 축으로 맞춰 비교하고, Train 전용(라벨 의존) 분석과 Test 전용(분포 차이) 진단을 한 번에 수행하도록 구성했습니다.

## 사용 가이드
- **Sampler 가중치**는 실험 기록상 power 0.85에서 최적을 보였으므로 기본값을 0.85로 고정했습니다.
- 모든 분석은 `data/raw/{train,test}` 의 원본 이미지를 그대로 사용하며, 학습/추론에서 썼던 리사이즈나 정규화가 관여하지 않도록 합니다.
- 산출물은 아래 경로 규칙으로 저장됩니다.
  - `reports/eda/train/`
  - `reports/eda/test/`
  - `reports/eda/compare/`
- 장시간이 걸릴 수 있는 지표(허프 기반 회전 추정, Morphological 텍스트 밀도 등)는 선택적 실행 플래그로 제어합니다.

In [1]:
import json
import math
import os
from collections import Counter
from concurrent.futures import ProcessPoolExecutor, as_completed
from functools import partial
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import ndimage
from scipy.stats import ks_2samp, wasserstein_distance
from skimage.measure import label, regionprops
from skimage.morphology import binary_closing, disk
from skimage.transform import radon
from tqdm.auto import tqdm

In [2]:
PROJECT_ROOT_ENV = os.environ.get('PROJECT_ROOT')
def _detect_project_root(start: Path) -> Path:
    if PROJECT_ROOT_ENV:
        env_path = Path(PROJECT_ROOT_ENV).resolve()
        if (env_path / 'data').exists():
            return env_path
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'data').exists() and (candidate / 'src').exists():
            return candidate
    return current

PROJECT_ROOT = _detect_project_root(Path.cwd())
RAW_TRAIN_DIR = PROJECT_ROOT / 'data' / 'raw' / 'train'
RAW_TEST_DIR = PROJECT_ROOT / 'data' / 'raw' / 'test'
TRAIN_META_PATH = PROJECT_ROOT / 'data' / 'raw' / 'train.csv'
FOLD_PATH = PROJECT_ROOT / 'splits' / 'fold_indices_v1.pkl'

EDA_OUTPUT_ROOT = PROJECT_ROOT / 'reports' / 'eda'
TRAIN_OUTPUT_DIR = EDA_OUTPUT_ROOT / 'train'
TEST_OUTPUT_DIR = EDA_OUTPUT_ROOT / 'test'
COMPARE_OUTPUT_DIR = EDA_OUTPUT_ROOT / 'compare'
COMPARE_OVERLAY_DIR = COMPARE_OUTPUT_DIR / 'overlays'
SAMPLE_BOARD_DIR = EDA_OUTPUT_ROOT / 'sample_boards'

for path in [TRAIN_OUTPUT_DIR, TEST_OUTPUT_DIR, COMPARE_OUTPUT_DIR, COMPARE_OVERLAY_DIR, SAMPLE_BOARD_DIR]:
    path.mkdir(parents=True, exist_ok=True)

WEIGHT_SAMPLER_POWER = 0.85

# 고비용 지표 실행 여부
ENABLE_ROTATION_RADON = True
ENABLE_QR_SEAL_SCORE = True
PROCESS_POOL_WORKERS = max(1, os.cpu_count() - 1)
BATCH_SIZE = 64  # ProcessPool 제출 단위

DISPLAY_INLINE_PLOTS = True  # set False to skip inline rendering when running headless


## 유틸 함수 정의
이미지 로딩부터 각종 지표 산출, 통계 요약, 시각화까지 재사용 가능한 함수로 분리했습니다. 동일한 배치/전처리를 강제하기 위해 OpenCV를 사용하고, 라벨 의존 로직은 Train 데이터 전용 함수에서만 접근하도록 구분합니다.

In [3]:
def load_image_bgr(path: Path) -> np.ndarray:
    """유니코드 경로 대응을 위해 imdecode 사용."""
    data = np.fromfile(str(path), dtype=np.uint8)
    image = cv2.imdecode(data, cv2.IMREAD_COLOR)
    return image


def clamp_ratio(value: float, eps: float = 1e-6) -> float:
    return float(value) if not np.isnan(value) else 0.0


def compute_margin_ratio(binary_mask: np.ndarray) -> float:
    filled = np.count_nonzero(binary_mask)
    total = binary_mask.size
    return clamp_ratio(1.0 - filled / max(total, 1))


def compute_rotation_offset(image_gray: np.ndarray) -> float:
    if not ENABLE_ROTATION_RADON:
        return 0.0
    theta = np.linspace(-15, 15, 181)
    sinogram = radon(image_gray, theta=theta, circle=False)
    projection = sinogram.sum(axis=0)
    angle = theta[int(np.argmax(projection))]
    return float(angle)

In [4]:
def compute_qr_seal_score(edge_map: np.ndarray) -> float:
    if not ENABLE_QR_SEAL_SCORE:
        return 0.0
    hough_circles = cv2.HoughCircles(
        edge_map,
        cv2.HOUGH_GRADIENT,
        dp=1.2,
        minDist=30,
        param1=100,
        param2=30,
        minRadius=8,
        maxRadius=60,
    )
    circle_count = 0 if hough_circles is None else hough_circles.shape[1]
    qr_density = np.sum(edge_map[:, ::4] > 0) / max(edge_map.size // 4, 1)
    return float(circle_count + qr_density)


def jpeg_blockiness_proxy(gray_image: np.ndarray) -> float:
    block = gray_image.astype(np.float32)
    vertical = np.mean(np.abs(block[:, 1:] - block[:, :-1])[:, ::8])
    horizontal = np.mean(np.abs(block[1:, :] - block[:-1, :])[::8, :])
    return float((vertical + horizontal) / 2.0)


def estimate_content_bbox(binary_mask: np.ndarray):
    labeled = label(binary_mask)
    if labeled.max() == 0:
        h, w = binary_mask.shape
        return (0, 0, w, h), 1.0, (0.5, 0.5)
    regions = sorted(regionprops(labeled), key=lambda r: r.area, reverse=True)
    main_region = regions[0]
    minr, minc, maxr, maxc = main_region.bbox
    region_area = main_region.area / binary_mask.size
    center_y = (minr + maxr) / 2 / binary_mask.shape[0]
    center_x = (minc + maxc) / 2 / binary_mask.shape[1]
    return (minc, minr, maxc, maxr), float(region_area), (float(center_x), float(center_y))

In [5]:
def compute_image_metrics(image_path: Path) -> dict:
    result = {
        "image_id": image_path.stem,
        "file_path": str(image_path),
        "file_size_bytes": image_path.stat().st_size,
        "load_error": False,
    }
    image = load_image_bgr(image_path)
    if image is None:
        result["load_error"] = True
        return result

    height, width = image.shape[:2]
    result.update({
        "width": width,
        "height": height,
        "aspect_ratio": clamp_ratio(width / max(height, 1)),
        "pixel_count": width * height,
    })

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    result["brightness_mean"] = float(gray.mean())
    result["brightness_std"] = float(gray.std())
    result["laplacian_var"] = float(cv2.Laplacian(gray, cv2.CV_64F).var())

    # 색상 정보
    b, g, r = cv2.split(image)
    channel_means = np.array([b.mean(), g.mean(), r.mean()])
    channel_stds = np.array([b.std(), g.std(), r.std()])
    rg = r.astype(np.float32) - g.astype(np.float32)
    yb = 0.5 * (r.astype(np.float32) + g.astype(np.float32)) - b.astype(np.float32)
    colorfulness = np.sqrt(rg.var()) + np.sqrt(yb.var())
    result["colorfulness"] = float(colorfulness)
    result["channel_mean_std"] = float(channel_stds.mean())
    result["gray_proxy"] = float(np.mean(np.abs(b.astype(np.float32) - r.astype(np.float32))) / 255.0)

    # 에지/텍스트 밀도
    edges = cv2.Canny(gray, 100, 200)
    result["edge_density"] = float(np.count_nonzero(edges) / max(edges.size, 1))
    qr_score = compute_qr_seal_score(edges)
    result["qr_seal_score"] = qr_score

    # 텍스트 밀도 proxy: Morphological closing 후 연결 성분
    _, otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    binary = (255 - otsu) > 0  # 텍스트를 True로
    closed = binary_closing(binary, disk(3))
    labeled = label(closed)
    components = regionprops(labeled)
    total_area = sum([region.area for region in components])
    result["text_component_count"] = len(components)
    result["text_area_ratio"] = clamp_ratio(total_area / max(binary.size, 1))

    # 문서 여백, 바운딩 박스, 회전
    doc_mask = binary.astype(np.uint8)
    margin_ratio = compute_margin_ratio(doc_mask)
    bbox, bbox_ratio, bbox_center = estimate_content_bbox(doc_mask)
    rot_offset = compute_rotation_offset(gray)
    result["margin_ratio"] = margin_ratio
    result["bbox_area_ratio"] = bbox_ratio
    result["bbox_center_x"] = bbox_center[0]
    result["bbox_center_y"] = bbox_center[1]
    result["rotation_offset_deg"] = rot_offset

    # JPEG Proxy
    result["jpeg_blockiness"] = jpeg_blockiness_proxy(gray)
    result["bytes_per_pixel"] = result["file_size_bytes"] / max(result["pixel_count"], 1)

    return result

In [6]:
def chunked_iterable(iterable, size):
    chunk = []
    for item in iterable:
        chunk.append(item)
        if len(chunk) == size:
            yield chunk
            chunk = []
    if chunk:
        yield chunk


def compute_dataset_frame(name: str, image_dir: Path, cache_path: Path) -> pd.DataFrame:
    if cache_path.exists():
        return pd.read_parquet(cache_path)

    image_paths = sorted(image_dir.glob('*.jpg'))
    records = []
    if PROCESS_POOL_WORKERS > 1:
        with ProcessPoolExecutor(max_workers=PROCESS_POOL_WORKERS) as executor:
            for result in tqdm(executor.map(compute_image_metrics, image_paths, chunksize=BATCH_SIZE),
                                total=len(image_paths), desc=f'metrics:{name}'):
                records.append(result)
    else:
        for path in tqdm(image_paths, desc=f'metrics:{name}'):
            records.append(compute_image_metrics(path))

    frame = pd.DataFrame.from_records(records)
    frame.to_parquet(cache_path, index=False)
    return frame

In [7]:
def overlay_histogram(train_df, test_df, column, bins=50, range_override=None, xlabel=None):
    plt.figure(figsize=(8, 4))
    common_kwargs = dict(bins=bins, alpha=0.5, density=True)
    if range_override:
        common_kwargs["range"] = range_override
    plt.hist(train_df[column].dropna(), label="Train", color="#1f77b4", **common_kwargs)
    plt.hist(test_df[column].dropna(), label="Test", color="#ff7f0e", **common_kwargs)
    plt.legend()
    plt.title(f"{column} distribution")
    plt.xlabel(xlabel or column)
    plt.ylabel("Density")
    plt.tight_layout()
    output_path = COMPARE_OVERLAY_DIR / f"{column}_overlay.png"
    plt.savefig(output_path, dpi=200)
    plt.close()
    return output_path


def resolution_heatmap(df, title, output_path):
    plt.figure(figsize=(6, 5))
    heatmap = plt.hist2d(df["width"], df["height"], bins=50, cmap="magma")
    plt.colorbar(heatmap[3])
    plt.title(title)
    plt.xlabel("Width")
    plt.ylabel("Height")
    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()
    return output_path


def boxplot_feature(train_df, test_df, column, ylabel=None):
    data = pd.DataFrame({"Dataset": ["Train"] * len(train_df) + ["Test"] * len(test_df), column: pd.concat([train_df[column], test_df[column]])})
    plt.figure(figsize=(6, 4))
    sns.boxplot(data=data, x="Dataset", y=column)
    plt.title(f"{column} boxplot")
    if ylabel:
        plt.ylabel(ylabel)
    plt.tight_layout()
    output_path = COMPARE_OVERLAY_DIR / f"{column}_boxplot.png"
    plt.savefig(output_path, dpi=200)
    plt.close()
    return output_path

In [ ]:
def resolution_heatmap_xy(df, xcol, ycol, title, output_path):
    plt.figure(figsize=(6, 5))
    heatmap = plt.hist2d(df[xcol], df[ycol], bins=50, cmap='magma')
    plt.colorbar(heatmap[3])
    plt.title(title)
    plt.xlabel(xcol)
    plt.ylabel(ycol)
    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    if 'DISPLAY_INLINE_PLOTS' in globals() and DISPLAY_INLINE_PLOTS:
        plt.show()
    plt.close()
    return output_path


In [8]:
def compute_distance_metrics(train_series, test_series):
    train_values = train_series.dropna().values
    test_values = test_series.dropna().values
    if len(train_values) == 0 or len(test_values) == 0:
        return {"ks_stat": np.nan, "ks_pvalue": np.nan, "wasserstein": np.nan}
    ks_stat, ks_p = ks_2samp(train_values, test_values)
    w_distance = wasserstein_distance(train_values, test_values)
    return {"ks_stat": float(ks_stat), "ks_pvalue": float(ks_p), "wasserstein": float(w_distance)}


def summarize_dataframe(df: pd.DataFrame, prefix: str, output_dir: Path) -> Path:
    summary = df.describe(include="all").transpose()
    summary_path = output_dir / f"{prefix}_summary.csv"
    summary.to_csv(summary_path)
    return summary_path

## 지표 계산 실행
Train/Test 각각의 이미지에 대해 고정된 지표를 계산합니다. 캐시 파일(`.parquet`)이 존재하면 재사용해 시간을 절약하도록 했습니다.

In [9]:
TRAIN_CACHE = TRAIN_OUTPUT_DIR / "train_image_metrics.parquet"
TEST_CACHE = TEST_OUTPUT_DIR / "test_image_metrics.parquet"

train_metrics = compute_dataset_frame("train", RAW_TRAIN_DIR, TRAIN_CACHE)
test_metrics = compute_dataset_frame("test", RAW_TEST_DIR, TEST_CACHE)

_ = summarize_dataframe(train_metrics, "train", TRAIN_OUTPUT_DIR)
_ = summarize_dataframe(test_metrics, "test", TEST_OUTPUT_DIR)

train_metrics.head(), test_metrics.head()

(           image_id                                          file_path  \
 0  002f99746285dfdd  /root/Computer_Vision_Contest-Private-/data/ra...   
 1  008ccd231e1fea5d  /root/Computer_Vision_Contest-Private-/data/ra...   
 2  008f5911bfda7695  /root/Computer_Vision_Contest-Private-/data/ra...   
 3  009235e4c9c07af5  /root/Computer_Vision_Contest-Private-/data/ra...   
 4  00b2f44967580c74  /root/Computer_Vision_Contest-Private-/data/ra...   
 
    file_size_bytes  load_error  width  height  aspect_ratio  pixel_count  \
 0           139275       False    443     591      0.749577       261813   
 1            70870       False    443     591      0.749577       261813   
 2            88669       False    443     591      0.749577       261813   
 3            69633       False    443     591      0.749577       261813   
 4           130417       False    443     591      0.749577       261813   
 
    brightness_mean  brightness_std  ...  qr_seal_score  text_component_count  \
 0 

In [ ]:
# Orientation-invariant sides and log10 pixel count
for _df in (train_metrics, test_metrics):
    _df['shorter_side'] = _df[['width','height']].min(axis=1)
    _df['longer_side']  = _df[['width','height']].max(axis=1)
    _df['pixel_count_log10'] = np.log10(_df['pixel_count'].replace(0, np.nan))

# Heatmaps on (shorter,longer)
train_sl = COMPARE_OVERLAY_DIR / 'train_shorter_longer_heatmap.png'
test_sl  = COMPARE_OVERLAY_DIR / 'test_shorter_longer_heatmap.png'
resolution_heatmap_xy(train_metrics, 'shorter_side','longer_side','Train shorter/longer', train_sl)
resolution_heatmap_xy(test_metrics,  'shorter_side','longer_side','Test shorter/longer',  test_sl)

# Log10 pixel count overlays
overlay_histogram(train_metrics, test_metrics, 'pixel_count_log10', bins=50, xlabel='log10(pixel_count)')


## Train 전용 메타데이터 결합
Fold 균형과 클래스 통계를 계산하기 위해 `train.csv`와 `fold_indices_v1.pkl`을 로드한 뒤, 앞서 계산한 지표와 병합합니다.

In [10]:
train_meta = pd.read_csv(TRAIN_META_PATH)
train_meta.rename(columns=lambda c: c.lower(), inplace=True)
if 'image_id' not in train_meta.columns:
    if 'id' in train_meta.columns:
        train_meta['image_id'] = train_meta['id'].apply(lambda x: Path(str(x)).stem)
    else:
        raise ValueError('train.csv 에 image_id 또는 id 컬럼이 필요합니다.')
label_column = None
for candidate in ['label', 'target', 'category']:
    if candidate in train_meta.columns:
        label_column = candidate
        break
if label_column is None:
    raise ValueError('train.csv 에 label/target/category 컬럼이 필요합니다.')
train_meta.rename(columns={label_column: 'label'}, inplace=True)
train_df = train_metrics.merge(train_meta[['image_id', 'label']], on='image_id', how='left')

if FOLD_PATH.exists():
    fold_obj = pd.read_pickle(FOLD_PATH)
    fold_map = {}
    if isinstance(fold_obj, dict):
        fold_iter = fold_obj.items()
    else:
        fold_iter = enumerate(fold_obj)
    for fold, entry in fold_iter:
        if isinstance(entry, (list, tuple)) and len(entry) >= 2:
            val_indices = entry[1]
        else:
            val_indices = entry
        if isinstance(val_indices, (list, tuple, np.ndarray)):
            val_ids = train_meta.iloc[list(val_indices)]['image_id']
        else:
            val_ids = [train_meta.iloc[int(val_indices)]['image_id']]
        for image_id in val_ids:
            fold_map[Path(str(image_id)).stem] = fold
    train_df['fold'] = train_df['image_id'].map(fold_map)

train_df.head()


,image_id,file_path,file_size_bytes,load_error,width,height,aspect_ratio,pixel_count,brightness_mean,brightness_std,...,text_area_ratio,margin_ratio,bbox_area_ratio,bbox_center_x,bbox_center_y,rotation_offset_deg,jpeg_blockiness,bytes_per_pixel,label,fold
0,002f99746285dfdd,/root/Computer_Vision_Contest-Private-/data/ra...,139275,False,443,591,0.749577,261813,112.680257,70.051393,...,0.667694,0.421671,0.389771,0.500000,0.500000,-13.666667,14.766545,0.531964,16,2
1,008ccd231e1fea5d,/root/Computer_Vision_Contest-Private-/data/ra...,70870,False,443,591,0.749577,261813,157.959471,29.099328,...,0.205895,0.898462,0.029498,0.496614,0.802876,-11.333333,6.962424,0.270689,10,2
2,008f5911bfda7695,/root/Computer_Vision_Contest-Private-/data/ra...,88669,False,443,591,0.749577,261813,160.815968,34.911001,...,0.231597,0.860003,0.052045,0.478555,0.447547,11.333333,9.854511,0.338673,10,4
3,009235e4c9c07af5,/root/Computer_Vision_Contest-Private-/data/ra...,69633,False,443,591,0.749577,261813,145.534313,38.807893,...,0.217212,0.845271,0.042893,0.480813,0.945854,-12.666667,5.361995,0.265965,4,5
4,00b2f44967580c74,/root/Computer_Vision_Contest-Private-/data/ra...,130417,False,443,591,0.749577,261813,101.450199,56.006780,...,0.688472,0.405091,0.434619,0.500000,0.664129,13.833333,9.776051,0.498130,16,1


## Train/Test 공통 지표 오버레이 및 거리 계산
같은 bin과 축으로 맞춘 히스토그램, 박스플롯을 생성하고 K-S, Wasserstein 거리를 표 형태로 기록합니다.

In [11]:
COMMON_FEATURES = [
    "width",
    "height",
    "aspect_ratio",
    "file_size_bytes",
    "pixel_count",
    "brightness_mean",
    "brightness_std",
    "laplacian_var",
    "colorfulness",
    "margin_ratio",
    "rotation_offset_deg",
    "edge_density",
    "text_area_ratio",
    "text_component_count",
    "qr_seal_score",
    "jpeg_blockiness",
    "bytes_per_pixel",
]

distance_records = []

def cohens_d(a, b):
    a = pd.Series(a).dropna().astype(float)
    b = pd.Series(b).dropna().astype(float)
    if len(a)<2 or len(b)<2:
        return float('nan')
    na, nb = len(a), len(b)
    va, vb = a.var(ddof=1), b.var(ddof=1)
    pooled = ((na-1)*va + (nb-1)*vb) / max(na+nb-2,1)
    if pooled<=0:
        return float('nan')
    return (a.mean()-b.mean())/np.sqrt(pooled)

for feature in tqdm(COMMON_FEATURES, desc="overlay"):
    overlay_path = overlay_histogram(train_metrics, test_metrics, feature)
    boxplot_path = boxplot_feature(train_metrics, test_metrics, feature)
    stats = compute_distance_metrics(train_metrics[feature], test_metrics[feature])
    # add directional stats
    tr = train_metrics[feature]; te = test_metrics[feature]
    stats.update({
        'mean_train': float(tr.mean()),
        'mean_test': float(te.mean()),
        'median_train': float(tr.median()),
        'median_test': float(te.median()),
        'mean_diff': float(te.mean()-tr.mean()),
        'median_diff': float(te.median()-tr.median()),
        'cohens_d': float(cohens_d(te, tr)),
    })
    stats.update({
        "feature": feature,
        "overlay_path": str(overlay_path.relative_to(PROJECT_ROOT)),
        "boxplot_path": str(boxplot_path.relative_to(PROJECT_ROOT)),
    })
    distance_records.append(stats)

distance_df = pd.DataFrame(distance_records).set_index("feature")
distance_path = COMPARE_OUTPUT_DIR / "stats_summary.json"
distance_df.to_json(distance_path, orient="index", indent=2)
distance_df


overlay:   0%|          | 0/17 [00:00<?, ?it/s]

,ks_stat,ks_pvalue,wasserstein,overlay_path,boxplot_path
feature,,,,,
width,0.167812,2.488922e-26,28.555442,reports/eda/compare/overlays/width_overlay.png,reports/eda/compare/overlays/width_boxplot.png
height,0.166556,6.078612e-26,28.448291,reports/eda/compare/overlays/height_overlay.png,reports/eda/compare/overlays/height_boxplot.png
aspect_ratio,0.167812,2.488922e-26,0.113853,reports/eda/compare/overlays/aspect_ratio_over...,reports/eda/compare/overlays/aspect_ratio_boxp...
file_size_bytes,0.176196,5.384904e-29,7573.831578,reports/eda/compare/overlays/file_size_bytes_o...,reports/eda/compare/overlays/file_size_bytes_b...
pixel_count,0.021447,7.077449e-01,7.112954,reports/eda/compare/overlays/pixel_count_overl...,reports/eda/compare/overlays/pixel_count_boxpl...
brightness_mean,0.319565,1.609639e-95,23.746778,reports/eda/compare/overlays/brightness_mean_o...,reports/eda/compare/overlays/brightness_mean_b...
brightness_std,0.039179,7.657473e-02,1.665606,reports/eda/compare/overlays/brightness_std_ov...,reports/eda/compare/overlays/brightness_std_bo...
laplacian_var,0.446828,8.610348e-190,665.593621,reports/eda/compare/overlays/laplacian_var_ove...,reports/eda/compare/overlays/laplacian_var_box...
colorfulness,0.175201,1.132878e-28,4.910426,reports/eda/compare/overlays/colorfulness_over...,reports/eda/compare/overlays/colorfulness_boxp...


## 해상도 2D 히트맵
Train과 Test 각각의 해상도 분포를 2D 히트맵으로 저장한 뒤, Markdown 리포트에서 나란히 비교할 수 있도록 합니다.

In [12]:
train_heatmap_path = COMPARE_OVERLAY_DIR / "train_resolution_heatmap.png"
test_heatmap_path = COMPARE_OVERLAY_DIR / "test_resolution_heatmap.png"

resolution_heatmap(train_metrics, "Train resolution", train_heatmap_path)
resolution_heatmap(test_metrics, "Test resolution", test_heatmap_path)

train_heatmap_path.relative_to(PROJECT_ROOT), test_heatmap_path.relative_to(PROJECT_ROOT)

(PosixPath('reports/eda/compare/overlays/train_resolution_heatmap.png'),
 PosixPath('reports/eda/compare/overlays/test_resolution_heatmap.png'))

## 클래스/폴드별 추가 분석 (Train 전용)
롱테일 분포 확인, 클래스별 평균 지표 레이더 차트, 폴드별 분포 차이를 계산합니다.

In [13]:
class_counts = train_df["label"].value_counts().sort_index()
plt.figure(figsize=(10, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, color="#1f77b4")
plt.title("Class distribution (Train)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
class_dist_path = TRAIN_OUTPUT_DIR / "class_distribution.png"
plt.savefig(class_dist_path, dpi=200)
plt.close()

class_means = train_df.groupby("label")[COMMON_FEATURES].mean()
train_overall_means = train_df[COMMON_FEATURES].mean()

radar_columns = ["brightness_mean", "brightness_std", "margin_ratio", "edge_density", "laplacian_var", "rotation_offset_deg"]
radar_angles = np.linspace(0, 2 * np.pi, len(radar_columns), endpoint=False)

def plot_radar_for_class(label_value, ax):
    values = class_means.loc[label_value, radar_columns]
    baseline = train_overall_means[radar_columns]
    values = np.concatenate([values.values, [values.values[0]]])
    baseline = np.concatenate([baseline.values, [baseline.values[0]]])
    angles = np.concatenate([radar_angles, [radar_angles[0]]])
    ax.plot(angles, values, label=f"Class {label_value}")
    ax.plot(angles, baseline, label="Train mean", linestyle="--")
    ax.fill(angles, values, alpha=0.2)
    ax.set_xticks(radar_angles)
    ax.set_xticklabels(radar_columns)
    ax.set_title(f"Class {label_value}")

radar_fig, radar_axes = plt.subplots(math.ceil(len(class_means) / 3), 3, subplot_kw={"projection": "polar"}, figsize=(15, max(6, 3 * math.ceil(len(class_means) / 3))))
radar_axes = radar_axes.flatten()

for idx, label_value in enumerate(class_means.index):
    plot_radar_for_class(label_value, radar_axes[idx])

for ax in radar_axes[len(class_means):]:
    ax.axis("off")

radar_axes[0].legend(loc="upper right", bbox_to_anchor=(1.3, 1.0))
radar_fig.suptitle("Class-wise feature radar vs Train mean", y=1.02)
radar_fig.tight_layout()
radar_path = TRAIN_OUTPUT_DIR / "class_radar_features.png"
radar_fig.savefig(radar_path, dpi=200, bbox_inches="tight")
plt.close(radar_fig)

# 폴드 분포 거리 측정
fold_distance_records = []
if "fold" in train_df.columns:
    for feature in COMMON_FEATURES:
        base_series = train_df[feature]
        for fold_value, fold_subset in train_df.groupby("fold"):
            distances = compute_distance_metrics(base_series, fold_subset[feature])
            distances.update({"feature": feature, "fold": fold_value})
            fold_distance_records.append(distances)

if fold_distance_records:
    fold_distance_df = pd.DataFrame(fold_distance_records)
    fold_distance_path = TRAIN_OUTPUT_DIR / "fold_distribution_distances.csv"
    fold_distance_df.to_csv(fold_distance_path, index=False)
    fold_distance_df.head()
else:
    fold_distance_df = pd.DataFrame()
    fold_distance_path = None
    fold_distance_df

## 혼동쌍 후보 탐색
클래스 평균 이미지를 만들고, SSIM 기반으로 근접도가 높은 클래스를 찾습니다. 시간이 오래 걸릴 수 있으므로 상위 k개만 계산하도록 제한합니다.

In [14]:
from skimage.metrics import structural_similarity as ssim

def load_gray(image_path: Path) -> np.ndarray:
    image = load_image_bgr(image_path)
    if image is None:
        return None
    return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

def compute_class_mean_image(df: pd.DataFrame, label_value, sample_limit=200) -> np.ndarray:
    subset = df[df["label"] == label_value].sample(n=min(sample_limit, len(df[df["label"] == label_value])), random_state=42)
    accum = None
    count = 0
    for _, row in subset.iterrows():
        gray = load_gray(Path(row["file_path"]))
        if gray is None:
            continue
        resized = cv2.resize(gray, (256, 256), interpolation=cv2.INTER_AREA)
        accum = resized.astype(np.float32) if accum is None else accum + resized.astype(np.float32)
        count += 1
    if accum is None or count == 0:
        return np.zeros((256, 256), dtype=np.uint8)
    mean_img = (accum / count).astype(np.uint8)
    return mean_img

class_mean_images = {label: compute_class_mean_image(train_df, label) for label in class_means.index}
ssim_records = []
labels = list(class_mean_images.keys())
for i, label_a in enumerate(labels):
    for label_b in labels[i + 1:]:
        score = ssim(class_mean_images[label_a], class_mean_images[label_b])
        ssim_records.append({"class_a": label_a, "class_b": label_b, "ssim": float(score)})

ssim_df = pd.DataFrame(ssim_records).sort_values("ssim", ascending=False)
ssim_path = TRAIN_OUTPUT_DIR / "confusable_pairs_ssim.csv"
ssim_df.to_csv(ssim_path, index=False)
ssim_df.head()

,class_a,class_b,ssim
53,3,12,0.896499
48,3,7,0.895063
45,3,4,0.894873
116,10,12,0.891990
51,3,10,0.888516


## 샘플 보드 생성
각 지표의 분위수(예: 10/50/90%)에 해당하는 이미지를 모아 시각적으로 바로 확인할 수 있도록 그리드를 생성합니다.

In [15]:
def generate_sample_board(df, feature, dataset_name, quantiles=(0.1, 0.5, 0.9), samples_per_quantile=5):
    fig, axes = plt.subplots(len(quantiles), samples_per_quantile, figsize=(samples_per_quantile * 2.5, len(quantiles) * 2.5))
    axes = np.atleast_2d(axes)
    for row_idx, q in enumerate(quantiles):
        target_value = df[feature].quantile(q)
        candidates = df.iloc[(df[feature] - target_value).abs().sort_values().index][:samples_per_quantile]
        for col_idx, (_, record) in enumerate(candidates.iterrows()):
            ax = axes[row_idx, col_idx]
            image = load_image_bgr(Path(record["file_path"]))
            if image is None:
                ax.axis("off")
                continue
            rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            ax.imshow(rgb)
            ax.axis("off")
            ax.set_title(f"Q{int(q*100)} | {record['image_id']}")
    fig.suptitle(f"{dataset_name} sample board · {feature}")
    fig.tight_layout()
    output_path = SAMPLE_BOARD_DIR / f"{dataset_name}_{feature}_board.png"
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return output_path

SAMPLE_FEATURES = ["rotation_offset_deg", "margin_ratio", "brightness_mean", "laplacian_var", "qr_seal_score"]
sample_board_manifest = []
for feature in SAMPLE_FEATURES:
    train_board = generate_sample_board(train_metrics, feature, "train")
    test_board = generate_sample_board(test_metrics, feature, "test")
    sample_board_manifest.append({
        "feature": feature,
        "train_board": str(train_board.relative_to(PROJECT_ROOT)),
        "test_board": str(test_board.relative_to(PROJECT_ROOT)),
    })

sample_board_manifest_path = SAMPLE_BOARD_DIR / "sample_boards_manifest.json"
with open(sample_board_manifest_path, "w", encoding="utf-8") as f:
    json.dump(sample_board_manifest, f, ensure_ascii=False, indent=2)

sample_board_manifest

[{'feature': 'rotation_offset_deg',
  'train_board': 'reports/eda/sample_boards/train_rotation_offset_deg_board.png',
  'test_board': 'reports/eda/sample_boards/test_rotation_offset_deg_board.png'},
 {'feature': 'margin_ratio',
  'train_board': 'reports/eda/sample_boards/train_margin_ratio_board.png',
  'test_board': 'reports/eda/sample_boards/test_margin_ratio_board.png'},
 {'feature': 'brightness_mean',
  'train_board': 'reports/eda/sample_boards/train_brightness_mean_board.png',
  'test_board': 'reports/eda/sample_boards/test_brightness_mean_board.png'},
 {'feature': 'laplacian_var',
  'train_board': 'reports/eda/sample_boards/train_laplacian_var_board.png',
  'test_board': 'reports/eda/sample_boards/test_laplacian_var_board.png'},
 {'feature': 'qr_seal_score',
  'train_board': 'reports/eda/sample_boards/train_qr_seal_score_board.png',
  'test_board': 'reports/eda/sample_boards/test_qr_seal_score_board.png'}]

## Train/Test 비교 리포트 자동 생성
`reports/eda/compare_train_test.md` 파일에 TL;DR, 주요 지표, 액션 체크리스트, 샘플 보드 링크를 기록합니다.

In [ ]:
# Augmentation simulation to match Test distribution (small sample)
import albumentations as A
from albumentations.pytorch import ToTensorV2

AUG_SAMPLE = len(train_metrics)
np.random.seed(42)
sel = train_metrics.sample(n=min(AUG_SAMPLE, len(train_metrics)), random_state=42)

aug = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=(0.0, 0.25), contrast_limit=(0.0, 0.1), p=0.6),
    A.GaussianBlur(blur_limit=(3,5), sigma_limit=(0.2,0.8), p=0.3),
    A.HueSaturationValue(hue_shift_limit=0, sat_shift_limit=(-12, 0), val_shift_limit=(0,10), p=0.4),
])

def recompute_metrics_aug(row):
    p=Path(row['file_path']); img=load_image_bgr(p)
    if img is None: return None
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    out = aug(image=img_rgb)['image']
    out_bgr = cv2.cvtColor(out, cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(out_bgr, cv2.COLOR_BGR2GRAY)
    rec, buf = cv2.imencode('.jpg', out_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 95])
    file_bytes = int(buf.size) if rec else row['file_size_bytes']
    h,w = gray.shape
    rec_dict = {
        'brightness_mean': float(gray.mean()),
        'brightness_std': float(gray.std()),
        'laplacian_var': float(cv2.Laplacian(gray, cv2.CV_64F).var()),
        'colorfulness': float((np.sqrt((out_bgr[:,:,2]-out_bgr[:,:,1]).var()) + np.sqrt(((0.5*(out_bgr[:,:,2]+out_bgr[:,:,1]) - out_bgr[:,:,0]).var())))),
        'bytes_per_pixel': file_bytes / float(h*w),
    }
    return rec_dict

aug_records=[]
for _, r in sel.iterrows():
    v=recompute_metrics_aug(r)
    if v is not None: aug_records.append(v)
train_aug = pd.DataFrame(aug_records)
train_aug.head()



In [ ]:
# Overlays: augmented Train vs Test for key features
AUG_OVER = COMPARE_OVERLAY_DIR / 'augmented'
AUG_OVER.mkdir(parents=True, exist_ok=True)
for feat in ['brightness_mean','laplacian_var','colorfulness','bytes_per_pixel']:
    overlay_histogram(train_aug.rename(columns={feat:feat}), test_metrics, feat)
train_aug.describe()


In [ ]:
# Parameter sweep to better match Test distribution (minimize sum of KS)
from itertools import product
def ks_sum(a, b, cols):
    s=0.0
    for c in cols:
        av=a[c].dropna().values; bv=b[c].dropna().values
        if len(av)==0 or len(bv)==0: continue
        s+=ks_2samp(av, bv)[0]
    return s

PARAM_GRID = {
  'bright_upper': [0.2, 0.25, 0.3],
  'blur_p': [0.25, 0.35, 0.45],
  'sigma_hi': [0.8, 1.0, 1.2],
  'sat_low': [0.8, 0.9, 1.0],
}
cols=['brightness_mean','laplacian_var','colorfulness','bytes_per_pixel']
best=None; best_params=None
for bu, bp, sh, sl in product(PARAM_GRID['bright_upper'], PARAM_GRID['blur_p'], PARAM_GRID['sigma_hi'], PARAM_GRID['sat_low']):
    aug = A.Compose([
        A.ColorJitter(brightness=(1.0, 1.0+bu), contrast=(0.9,1.1), saturation=(sl,1.0), hue=0, p=0.6),
        A.GaussianBlur(blur_limit=(3,7), sigma_limit=(0.2, sh), p=bp),
    ])
    recs=[]
    for _, r in sel.iterrows():
        p=Path(r['file_path']); data=np.fromfile(str(p), dtype=np.uint8); img=cv2.imdecode(data, cv2.IMREAD_COLOR)
        if img is None: continue
        out=aug(image=cv2.cvtColor(img, cv2.COLOR_BGR2RGB))['image']
        out_bgr=cv2.cvtColor(out, cv2.COLOR_RGB2BGR)
        gray=cv2.cvtColor(out_bgr, cv2.COLOR_BGR2GRAY)
        ok, buf=cv2.imencode('.jpg', out_bgr, [int(cv2.IMWRITE_JPEG_QUALITY),95])
        size=int(buf.size) if ok else int(r['file_size_bytes'])
        b,g,rr=cv2.split(out_bgr)
        rg=(rr.astype(np.float32)-g.astype(np.float32)); yb=0.5*(rr.astype(np.float32)+g.astype(np.float32))-b.astype(np.float32)
        color=np.sqrt(rg.var())+np.sqrt(yb.var())
        recs.append({
            'brightness_mean': float(gray.mean()),
            'laplacian_var': float(cv2.Laplacian(gray, cv2.CV_64F).var()),
            'colorfulness': float(color),
            'bytes_per_pixel': size/float(gray.size)
        })
    augdf=pd.DataFrame(recs)
    score=ks_sum(augdf, test_metrics, cols)
    if (best is None) or (score<best):
        best=score; best_params={'bright_upper':bu, 'blur_p':bp, 'sigma_hi':sh, 'sat_low':sl}

best, best_params


In [ ]:
# Generate overlays with best params and save tuned params
AUG_OVER = COMPARE_OVERLAY_DIR / 'augmented_tuned'
AUG_OVER.mkdir(parents=True, exist_ok=True)
bu=best_params['bright_upper']; bp=best_params['blur_p']; sh=best_params['sigma_hi']; sl=best_params['sat_low']
aug = A.Compose([
    A.ColorJitter(brightness=(1.0, 1.0+bu), contrast=(0.9,1.1), saturation=(sl,1.0), hue=0, p=0.6),
    A.GaussianBlur(blur_limit=(3,7), sigma_limit=(0.2, sh), p=bp),
])
recs=[]
for _, r in sel.iterrows():
    p=Path(r['file_path']); data=np.fromfile(str(p), dtype=np.uint8); img=cv2.imdecode(data, cv2.IMREAD_COLOR)
    if img is None: continue
    out=aug(image=cv2.cvtColor(img, cv2.COLOR_BGR2RGB))['image']
    out_bgr=cv2.cvtColor(out, cv2.COLOR_RGB2BGR)
    gray=cv2.cvtColor(out_bgr, cv2.COLOR_BGR2GRAY)
    ok, buf=cv2.imencode('.jpg', out_bgr, [int(cv2.IMWRITE_JPEG_QUALITY),95])
    size=int(buf.size) if ok else int(r['file_size_bytes'])
    b,g,rr=cv2.split(out_bgr); rg=(rr.astype(np.float32)-g.astype(np.float32)); yb=0.5*(rr.astype(np.float32)+g.astype(np.float32))-b.astype(np.float32)
    color=np.sqrt(rg.var())+np.sqrt(yb.var())
    recs.append({'brightness_mean': float(gray.mean()), 'laplacian_var': float(cv2.Laplacian(gray, cv2.CV_64F).var()), 'colorfulness': float(color), 'bytes_per_pixel': size/float(gray.size)})
augdf=pd.DataFrame(recs)
for feat in ['brightness_mean','laplacian_var','colorfulness','bytes_per_pixel']:
    plt.figure(figsize=(8,4))
    plt.hist(augdf[feat].dropna(), bins=50, alpha=0.5, density=True, label='Train_AUG_TUNED')
    plt.hist(test_metrics[feat].dropna(), bins=50, alpha=0.5, density=True, label='Test')
    plt.legend(); plt.title(f'{feat} (aug tuned vs test)'); plt.tight_layout()
    plt.savefig(AUG_OVER / f'{feat}_aug_tuned_overlay.png', dpi=200); plt.close()
json.dump({'best_score_ks_sum': best, **best_params}, open(COMPARE_OVERLAY_DIR / 'aug_tuned_params.json','w'))
AUG_OVER, COMPARE_OVERLAY_DIR / 'aug_tuned_params.json'


In [16]:
ACTION_RULES = [
    ("rotation_offset_deg", 0.2, "Test 회전 분포가 ±8° 이상 치우치면 Rotate 증강 상향 및 orientation normalize 고려"),
    ("margin_ratio", 0.1, "Test 여백비가 커지면 CenterCrop 대신 Pad 또는 RRC 비중 조정"),
    ("laplacian_var", 0.0, "Blur 증가 시 Blur 증강 on + 추론 시 Sharpen 비활성"),
    ("jpeg_blockiness", 0.0, "압축 강하면 JPEGCompression 증강 on, Swin late fusion 가중 ↑"),
    ("brightness_mean", 0.25, "밝기 편차 크면 ColorJitter 범위 확장"),
    ("aspect_ratio", 0.2, "Aspect Ratio 차이가 크면 longer-side resize + pad 일관화"),
    ("text_area_ratio", 0.1, "텍스트 밀도 감소 시 OCR 가산치 조정 및 레이아웃 모델 비중 확대"),
]

def format_action_lines(distance_df: pd.DataFrame) -> list:
    lines = []
    for feature, threshold, guidance in ACTION_RULES:
        if feature not in distance_df.index:
            continue
        ks_stat = distance_df.loc[feature, "ks_stat"]
        w1 = distance_df.loc[feature, "wasserstein"]
        if (not math.isnan(ks_stat) and ks_stat >= threshold) or (not math.isnan(w1) and w1 > 0):
            lines.append(f"- **{feature}**: KS={ks_stat:.3f}, W1={w1:.3f} → {guidance}")
    return lines

def summarize_top_differences(distance_df: pd.DataFrame, top_k=5) -> list:
    ranked = distance_df.copy()
    ranked["ks_abs"] = ranked["ks_stat"].abs()
    ranked["w1_abs"] = ranked["wasserstein"].abs()
    ranked.sort_values(["ks_abs", "w1_abs"], ascending=False, inplace=True)
    lines = []
    for feature, row in ranked.head(top_k).iterrows():
        lines.append(f"- **{feature}**: KS={row['ks_stat']:.3f} (p={row['ks_pvalue']:.3f}), W1={row['wasserstein']:.3f}")
    return lines

report_lines = [
    "# Train vs Test EDA Report",
    "\n",
    "## TL;DR",
]

report_lines.extend(summarize_top_differences(distance_df))
report_lines.append("\n## 바로 적용 액션")
action_lines = format_action_lines(distance_df)
if action_lines:
    report_lines.extend(action_lines)
else:
    report_lines.append("- 현재 임계선을 넘은 지표 없음")

report_lines.append("\n## 수치 요약")
report_lines.append("Feature | KS | KS p-value | W1")
report_lines.append("--- | --- | --- | ---")
for feature, row in distance_df.iterrows():
    report_lines.append(f"{feature} | {row['ks_stat']:.4f} | {row['ks_pvalue']:.4f} | {row['wasserstein']:.4f}")

report_lines.append("\n## 주요 플롯")
for feature, row in distance_df.iterrows():
    report_lines.append(f"- ![]({row['overlay_path']})")

report_lines.append("\n## 샘플 보드")
for entry in sample_board_manifest:
    report_lines.append(f"- {entry['feature']}: Train ![]({entry['train_board']}), Test ![]({entry['test_board']})")

report_lines.append("\n## 품질 체크리스트")
quality_checks = [
    "- 동일 bin/축 적용 여부 재확인",
    "- 분석용 로더가 학습/추론 전처리와 동일한지",
    "- 샘플러 가중치(0.85)가 EDA 통계에 영향을 주지 않는지",
    "- 손상 파일/중복 이미지 탐지 수행",
    "- Fold 분포와 Train 전체 분포 차이 점검",
]
report_lines.extend(quality_checks)

report_lines.append("\n## 추가 실험 제안")
report_lines.append("- A/B: Test 특이 분포(회전, 여백) 반영한 증강 vs 기존 증강")
report_lines.append("- A/B: JPEGCompression + Blur 증강 ON/OFF")
report_lines.append("- A/B: Orientation normalize 후 추론 TTA")

report_path = EDA_OUTPUT_ROOT / "compare_train_test.md"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

report_path.relative_to(PROJECT_ROOT)

PosixPath('reports/eda/compare_train_test.md')

## 무결성 점검 체크리스트
EDA 이전/이후로 반드시 실행할 간단한 자동화 스크립트를 포함합니다.

In [17]:
def detect_broken_images(df: pd.DataFrame) -> pd.DataFrame:
    broken = df[df["load_error"] == True]
    broken_path = EDA_OUTPUT_ROOT / "broken_images.csv"
    broken.to_csv(broken_path, index=False)
    return broken


def detect_duplicate_files(df: pd.DataFrame) -> pd.DataFrame:
    hash_records = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="hash"):
        file_path = Path(row["file_path"])
        data = np.fromfile(str(file_path), dtype=np.uint8)
        hash_records.append({
            "image_id": row["image_id"],
            "file_path": row["file_path"],
            "hash": hash(data.tobytes()).to_bytes(8, byteorder="little", signed=True).hex(),
        })
    hash_df = pd.DataFrame(hash_records)
    dupes = hash_df[hash_df.duplicated("hash", keep=False)].sort_values("hash")
    duplicate_path = EDA_OUTPUT_ROOT / "duplicates.csv"
    dupes.to_csv(duplicate_path, index=False)
    return dupes


broken_train = detect_broken_images(train_metrics)
broken_test = detect_broken_images(test_metrics)
duplicate_candidates = detect_duplicate_files(pd.concat([train_metrics, test_metrics], ignore_index=True))

broken_train, broken_test, duplicate_candidates.head()

hash:   0%|          | 0/4721 [00:00<?, ?it/s]

(Empty DataFrame
 Columns: [image_id, file_path, file_size_bytes, load_error, width, height, aspect_ratio, pixel_count, brightness_mean, brightness_std, laplacian_var, colorfulness, channel_mean_std, gray_proxy, edge_density, qr_seal_score, text_component_count, text_area_ratio, margin_ratio, bbox_area_ratio, bbox_center_x, bbox_center_y, rotation_offset_deg, jpeg_blockiness, bytes_per_pixel]
 Index: []
 
 [0 rows x 25 columns],
 Empty DataFrame
 Columns: [image_id, file_path, file_size_bytes, load_error, width, height, aspect_ratio, pixel_count, brightness_mean, brightness_std, laplacian_var, colorfulness, channel_mean_std, gray_proxy, edge_density, qr_seal_score, text_component_count, text_area_ratio, margin_ratio, bbox_area_ratio, bbox_center_x, bbox_center_y, rotation_offset_deg, jpeg_blockiness, bytes_per_pixel]
 Index: []
 
 [0 rows x 25 columns],
 Empty DataFrame
 Columns: [image_id, file_path, hash]
 Index: [])

## 정리
- Train/Test 지표를 동일한 방법으로 계산·시각화하여 분포 차이를 정량화했습니다.
- KS, Wasserstein 통계를 기반으로 즉각적인 증강/추론 조정 액션을 도출했습니다.
- 샘플 보드, 레이더 차트, 혼동쌍 분석으로 향후 모델링 전략(예: 클래스별 커스텀 증강, OCR 가중치 조정 등)을 설계할 수 있습니다.
- 모든 산출물은 `reports/eda/` 하위 경로에 저장되며, 변경 사항을 Git으로 추적해 서버 장애 후에도 같은 분석을 복원할 수 있습니다.